In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pickle
# from predictor_tf import load_eqcct_model
from predictor_pt import EQCCTModelS
# from predictor_pt_p import EQCCTModelP
import json
from loader import load_keras_h5_as_numpy_dict, load_torchish_into_eqcctS
from catalog import flatten_h5dict
from torch_weight_dict import build_torchish_state

In [2]:
tf_weights = load_keras_h5_as_numpy_dict("ModelPS/test_trainer_021.h5")
flat = flatten_h5dict(tf_weights)
torchish = build_torchish_state(flat)

— sanity check —
PatchEncoder: kernel 1600x40, bias 40, pos-emb 150x40
Heads present: picker_S=True, picker_P=False
picker_S: kernel 15x1x1, bias 1
Counts — conv1d: 33 kernels, bn: 33 γ/33 β, dense: 9, ln: 9
conv1d kernel shapes (first few): (11, 3, 3)×2, (11, 40, 40)×24, (11, 3, 10)×1, (11, 10, 10)×2, (11, 10, 20)×1, (11, 20, 20)×2
MHA blocks found: 4
  multi_head_attention: Q/K kernel (40, 4, 40), V kernel (40, 4, 40), out kernel (4, 40, 40)
  multi_head_attention_1: Q/K kernel (40, 4, 40), V kernel (40, 4, 40), out kernel (4, 40, 40)
  multi_head_attention_2: Q/K kernel (40, 4, 40), V kernel (40, 4, 40), out kernel (4, 40, 40)
  multi_head_attention_3: Q/K kernel (40, 4, 40), V kernel (40, 4, 40), out kernel (4, 40, 40)
— sanity ok —


In [3]:

# quick spot-check
for k in sorted([kk for kk in torchish if '.out_proj.weight' in kk])[:2]:
    print(k, torchish[k].shape)  # expect (E, H*D) e.g. (40, 160)

for k in sorted([kk for kk in torchish if '.q_proj.weight' in kk])[:2]:
    print(k, torchish[k].shape)  # expect (H*D, E) e.g. (160, 40)

multi_head_attention.out_proj.weight (40, 160)
multi_head_attention_1.out_proj.weight (40, 160)
multi_head_attention.q_proj.weight (160, 40)
multi_head_attention_1.q_proj.weight (160, 40)


In [4]:
model = EQCCTModelS()
model = load_torchish_into_eqcctS(model, torchish)

KeyError: "Missing PatchEncoder weight. Nearby: ['patch_encoder.linear.weights', 'patch_encoder.linear.bias', 'patch_encoder.linear.pos_embedding'] ..."